# MLOps Sentiment Analysis


Questo notebook illustra l'implementazione di un sistema MLOps completo per l'analisi del sentiment sui social media, realizzato per il corso **MLOps e Machine Learning in produzione**.

Il progetto consta di tre elementi cardine:
- **Modello Base** — Il modello `cardiffnlp/twitter-roberta-base-sentiment-latest` fornisce una baseline solida con accuracy ~72% e Macro F1 ~0.72 su `tweet_eval/sentiment`.

- **CI/CD** — La pipeline CI/CD con GitHub Actions automatizza training, validazione e deploy. Il model validation gate garantisce che venga deployato un modello migliore rispetto a quello in produzione.

- **Monitoraggio Continuo** — Il sistema di monitoraggio rileva automaticamente distribution shift e data drift tramite PSI. Ciò rende possibile monitorare in tempo reale lo stato del modello, con la possibilità di avviare un retraining del modello in produzione manualmente.

**Repository GitHub**: [https://github.com/giovannibonisoli/mlops-sentiment-analysis](https://github.com/giovannibonisoli/mlops-sentiment-analysis)

## Setup
Installazione delle dipendenze e clone del repository.

In [1]:
!pip install transformers torch datasets scikit-learn huggingface_hub numpy

In [14]:
!git clone https://github.com/giovannibonisoli/mlops-sentiment-analysis.git

fatal: destination path 'mlops-sentiment-analysis' already exists and is not an empty directory.


In [13]:
import sys
sys.path.insert(0, '/content/mlops-sentiment-analysis')

---
## Test del modello in produzione
In questa sezione viene caricato il modello deployato su HuggingFace Hub e testato su alcuni esempi rappresentativi. Questo verifica che il deploy sia andato a buon fine e che il modello risponda correttamente in un contesto simile a quello di produzione.

### Caricamento dataset e modello in produzione

Carica il modello deployato su HuggingFace Hub e testa alcune predizioni
per verificare che il deploy sia andato a buon fine.

In [25]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset('tweet_eval', 'sentiment')
test_data = dataset['test']

LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}
distribution = Counter(test_data['label'])
total = sum(distribution.values())

print(f'Dimensione test set: {len(test_data)}')
print('\nDistribuzione test set:')
for label_id, count in sorted(distribution.items()):
    print(f'{LABEL_MAP[label_id]}: {count} ({count/total:.1%})')

Dimensione test set: 12284

Distribuzione test set:
negative: 3972 (32.3%)
neutral: 5937 (48.3%)
positive: 2375 (19.3%)


### Caricamento del modello in produzione

Il modello viene caricato direttamente da HuggingFace Hub tramite `load_classifier()`. In produzione questo è il modello che verrebbe interrogato per classificare i testi provenienti dai social media monitorati.

In [16]:
from src.model import load_classifier, predict

In [17]:
HF_REPO = "giovannibonisoli/sentiment-model"
print(f"Caricamento modello in produzione da {HF_REPO}...")
prod_classifier = load_classifier(model_path=HF_REPO)

Caricamento modello in produzione da giovannibonisoli/sentiment-model...


config.json:   0%|          | 0.00/973 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vengono testati sei testi con sentiment volutamente distinti, due positivi, due negativi e due neutrali, per verificare che il modello in produzione classifichi correttamente le tre classi.


In [19]:
test_texts = [
    "I absolutely love this company, their service is outstanding!",
    "Terrible experience, I will never buy from them again.",
    "The product arrived on time, nothing special to report.",
    "This is the best purchase I have ever made!",
    "Very disappointed with the quality, expected much better.",
    "It works as described, does the job."
]

In [20]:
for text in test_texts:
    result = predict(prod_classifier, text)
    print(f"Text:  {text}")
    print(f"Label: {result[0]}")
    print()


=== Predizioni modello in produzione ===

Text:  I absolutely love this company, their service is outstanding!
Label: positive

Text:  Terrible experience, I will never buy from them again.
Label: negative

Text:  The product arrived on time, nothing special to report.
Label: positive

Text:  This is the best purchase I have ever made!
Label: positive

Text:  Very disappointed with the quality, expected much better.
Label: negative

Text:  It works as described, does the job.
Label: neutral



### Valutazione su subsample del dataset tweet_eval/sentiment

La valutazione viene effettuata su un campione casuale stratificato di 500 esempi dal test set di `tweet_eval/sentiment`. L'intero test set (~12k esempi) richiederebbe tempi eccessivi su Colab senza GPU.

In [28]:
test_sample = test_data.shuffle(seed=42).select(range(500))

y_true = [LABEL_MAP[l] for l in test_sample['label']]
y_pred = predict(prod_classifier, list(test_sample['text']))

print('=== Valutazione Baseline (500 esempi) ===')
print(classification_report(y_true, y_pred))
print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print(f'Macro F1: {f1_score(y_true, y_pred, average="macro"):.4f}')

=== Valutazione Baseline (500 esempi) ===
              precision    recall  f1-score   support

    negative       0.71      0.83      0.76       163
     neutral       0.73      0.63      0.68       223
    positive       0.75      0.78      0.77       114

    accuracy                           0.73       500
   macro avg       0.73      0.75      0.74       500
weighted avg       0.73      0.73      0.73       500

Accuracy: 0.7280
Macro F1: 0.7354


## Pipeline CI/CD

### Architettura della pipeline

La pipeline CI/CD è implementata con GitHub Actions in un singolo workflow (`CI_CD.yml`) con due job separati:

**Job 1 — Test** (eseguito ad ogni push/PR):
- Lint con flake8
- Unit test e integration test con pytest

**Job 2 — Train, Validate e Deploy** (eseguito solo da `workflow_dispatch`):
- Fine-tuning del modello su `tweet_eval/sentiment`
- Validazione: confronto Macro F1 con modello in produzione su HuggingFace
- Deploy su HuggingFace Hub solo se il nuovo modello è migliore

Questa separazione garantisce che l'allenamento e il deploy di un modello avvengano esclusivamente a su trigger manuale, e non ad ogni push.


## Monitoraggio Continuo

### Sistema di Monitoraggio

Il sistema di monitoraggio è composto da tre moduli:

- **`monitor.py`**: logga ogni predizione con timestamp, label e confidence in un CSV,
costruendo uno storico delle predizioni nel tempo.

- **`drift.py`**: analizza le predizioni e rileva tre tipi di anomalie:
  - **Distribution shift** — confronta la distribuzione corrente delle classi sentiment
  (positive/neutral/negative) con la baseline. Uno shift superiore al 15% su una classe
  indica un cambiamento significativo nel comportamento degli utenti, ad esempio un picco
  di negatività durante una crisi reputazionale.
  - **Confidence drift** — monitora la confidence media del modello. Un calo sotto la soglia
  di 0.65 indica che il modello è sistematicamente incerto sui nuovi testi, segnale precoce
  di cambiamento nel linguaggio o nel dominio dei dati.
  - **Data drift (PSI)** — calcola il Population Stability Index sulle distribuzioni delle
  confidence scores tra una finestra temporale e la precedente. PSI > 0.2 indica un
  cambiamento strutturale nei dati in ingresso, rilevabile prima ancora che le label
  cambino in modo evidente.

- **`simulate.py`**: simula l'arrivo di nuovi testi dai social media con tre modalità:
  - `normal` — distribuzione bilanciata da `tweet_eval`, rappresenta il funzionamento ordinario
  - `drift` — sovracampiona la classe negative (70% vs 32% baseline), simula una crisi
  reputazionale e garantisce il rilevamento del distribution shift
  - `data_drift` — usa recensioni cinematografiche da `imdb`, simula un domain shift dove
  il modello riceve testi strutturalmente diversi da quelli su cui è stato addestrato,
  abbassando la confidence e triggerando il PSI

In produzione `monitor.py` verrebbe alimentato da testi reali provenienti dalle API
dei social media monitorati (Twitter/X, Reddit, ecc.), sostituendo completamente la simulazione.

### Analisi del log di monitoraggio

In [33]:
import csv
from collections import Counter

rows = []
with open('/content/mlops-sentiment-analysis/predictions_log/predictions_log.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

distribution = Counter(r['predicted_label'] for r in rows)
total = sum(distribution.values())
confidences = [float(r['confidence']) for r in rows]

print(f'Totale predizioni loggiate: {len(rows)}')
print('\nDistribuzione complessiva:')
for label, count in distribution.items():
    print(f'  {label}: {count} ({count/total:.1%})')
print(f'\nConfidence media: {sum(confidences)/len(confidences):.4f}')

Totale predizioni loggiate: 300

Distribuzione complessiva:
  positive: 101 (33.7%)
  neutral: 83 (27.7%)
  negative: 116 (38.7%)

Confidence media: 0.7555


## Conclusioni

Il progetto implementa un sistema MLOps completo per l'analisi del sentiment sui social media, basata sul seguente ciclo:

```
Dati social → Predizione → Log → Monitoraggio → [Drift?] → Retraining → Nuovo Deploy
```

**Limiti**:
- La pipeline di riallenamento, validazione e deploy è lanciabile solo manualmente.
- I dati per il riallenamento per il momento vengono da dataset opensource
- Il sistema di monitoraggio si basa su dei CSV temporanei.

**Sviluppi futuri**
- Il retraining può essere parzialmente o completamente automatizzato a seconda del contesto.
- Alimentare il sistema con API reali dei social media
- Il retraining userebbe nuovi dati etichettati raccolti dal monitoraggio
- Il sistema di monitoraggio potrebbe essere esteso con un database persistente.